1. 두 벡터의 내적과 사이각(도 단위)을 구하는 코드를 작성하고, 손으로 계산한 값과 일치하는지 검증하세요.  
시드 난수를 활용해 미지의 테스트 벡터 a, b를 생성하고, 직접 구현한 내적 공식 및 사이각 공식이 대수적 손계산 수식과 일치하는지 일반화하여 검증

In [46]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(),"..")))

import numpy as np
import src.vectors as vec

rng = np.random.default_rng(42)
a = rng.uniform(-10, 10, 3)
b = rng.uniform(-10, 10, 3)

dot_res = vec.dot_product(a, b)
angle_res = vec.angle_between(a, b)
print(f"내적: {dot_res} / 사이각: {angle_res:.2f} 도")

내적: 99.77274599749983 / 사이각: 33.34 도


In [47]:
hand_dot = a[0]*b[0] + a[1]*b[1] + a[2]*b[2]
assert np.isclose(dot_res, hand_dot)
assert np.isclose(dot_res, np.dot(a, b)) # [검산] 내장함수 비교명시
print("문항 1 통과 (True)")

문항 1 통과 (True)


2. 벡터를 정규화하는 함수를 만들고, 영벡터를 넣으면 어떻게 되는지 직접 실행해 결과를 기록한 뒤, 어떻게 처리할지 정해 구현하세요.  
영벡터 입력 시 원래 공식대로라면 벡터의 크기인 분모가 0이 되어 오류가 발생하지만 vectors.py의 함수 normalize에서 영벡터 입력시 그대로 영백터를 반환하도록 처리함.

In [48]:
zero_v = np.array([0.0, 0.0, 0.0])
norm_zero = vec.normalize(zero_v)

In [49]:
assert np.allclose(norm_zero, np.array([0.0, 0.0, 0.0]))
print("문항 2 통과 (True)")

문항 2 통과 (True)


3. 한 벡터를 다른 벡터 방향으로 투영하는 project(a, b) 를 구현하고, ① 남는 성분이 b 와 수직인지 ② 두 성분을 더하면 원래 벡터가 되는지를 각각 검증하세요.  
벡터 a를 b에 투영한 project(a, b)와 투영 성분이 제거된 수직 오차 벡터 reject(a, b)를 구하고 남는 성분이 b와 수직인지, 두 성분을 더하면 기존 벡터인 a가 되는지 검증

In [50]:
project = vec.project(a, b)
reject = vec.reject(a, b)

In [51]:
is_orthogonal = np.isclose(vec.dot_product(reject, b), 0.0)
is_restored = np.allclose(project + reject, a)
assert is_orthogonal and is_restored
print(f"정사영 검증: 수직성 {is_orthogonal} / 합 복원 {is_restored}")

정사영 검증: 수직성 True / 합 복원 True


4. 외적을 반대칭행렬 곱으로 구현하는 skew(a) 를 작성해, skew(a) @ b 가 np.cross(a, b) 와 같은지 확인하고 skew(a) 가 실제 반대칭인지(전치하면 부호가 바뀌는지) 검증하세요.  
외적 연산의 행렬 치환 정리인 $\mathbf{a} \times \mathbf{b} = [\mathbf{a}]_\times \mathbf{b}$ 가 성립하는지 검증하고, 전치 시 부호가 반전되는 반대칭 성질 ($\mathbf{S}^T = -\mathbf{S}$)을 검증

In [52]:
S = vec.skew(a)
skew_cross = S @ b
direct_cross = vec.custom_cross(a, b)

In [53]:
assert np.allclose(skew_cross, direct_cross)
assert np.allclose(S.T, -S)
assert np.allclose(skew_cross, np.cross(a, b)) # [검산] 내장함수 비교명시
print("문항 4 통과 (True)")

문항 4 통과 (True)


5. 세 점으로 이루어진 평면의 단위 법선 벡터를 구하는 plane_normal(P1, P2, P3) 를 작성하세요.  
3차원 공간 상에 정렬되지 않은 임의의 세 점 $P_1, P_2, P_3$가 주어졌을 때, 이 세 점이 이루는 평면 위의 두 사이 벡터는 $\mathbf{v}_{12} = P_2 - P_1$, $\mathbf{v}_{13} = P_3 - P_1$ 이고, 두 벡터의 외적으로 그 평면에 수직인 법선 벡터를 얻을 수 있다. 이를 정규화하여 단위벡터를 도출

In [54]:
rng = np.random.default_rng(42)
p1 = rng.uniform(-10, 10, 3)
p2 = rng.uniform(-10, 10, 3)
p3 = rng.uniform(-10, 10, 3)

n_vector = vec.plane_normal(p1, p2, p3)

In [55]:
# ① 추출된 법선 벡터가 크기가 1인 단위 벡터인지 확인
is_unit_vector = np.isclose(vec.custom_norm(n_vector), 1.0)

# ② 법선 벡터는 평면 위의 두 사이 벡터(p2-p1, p3-p1)와 각각 반드시 수직(내적=0)이어야 함
v12 = p2 - p1
v13 = p3 - p1
is_ortho1 = np.isclose(vec.dot_product(n_vector, v12), 0.0)
is_ortho2 = np.isclose(vec.dot_product(n_vector, v13), 0.0)

assert is_unit_vector, "단위 법선 벡터의 크기가 1이 아닙니다."
assert is_ortho1 and is_ortho2, "도출된 법선 벡터가 평면과 수직을 이루지 않습니다."
print("통과: 문항 5 평면 단위 법선 벡터 및 수직성 검증 완료 (True)")

통과: 문항 5 평면 단위 법선 벡터 및 수직성 검증 완료 (True)


6. 세 벡터 (1,0,1), (0,1,1), (1,1,2)의 rank를 구하고 왜 3이 아닌지를 벡터 사이 관계로 설명하세요. 같은 행렬의 행렬식도 구해 두 결과가 일관되는지 확인합니다.  
세 벡터 $\mathbf{v}_1=(1,0,1), \mathbf{v}_2=(0,1,1), \mathbf{v}_3=(1,1,2)$ 사이의 대수적 선형 관계를 살펴보면, $\mathbf{v}_1 + \mathbf{v}_2 = (1+0, 0+1, 1+1) = (1,1,2) = \mathbf{v}_3$ 가 정확히 성립. 세 벡터는 선형 종속 상태이므로 이 때의 Rank는 2인 특이행렬이 되고 행렬식도 0이 된다. 이 두 성질의 일관성을 검증.

In [56]:
M = np.array([
    [1.0, 0.0, 1.0],
    [0.0, 1.0, 1.0],
    [1.0, 1.0, 2.0]
], dtype=float)

rank_val = np.linalg.matrix_rank(M)
det_val = np.linalg.det(M)

In [57]:
assert rank_val == 2, "Rank가 2가 아닙니다 (선형 종속성 검증 실패)."
assert np.isclose(det_val, 0.0), "행렬식이 수치적으로 0이 아닙니다 (부피 형성 실패)."
print(f"통과 - 세 벡터의 rank: {rank_val} / 행렬식 값: {det_val:.1f} (대수적 일관성 일치)")

통과 - 세 벡터의 rank: 2 / 행렬식 값: 0.0 (대수적 일관성 일치)


7. 완성한 함수들을 src/vectors.py 로 내보내세요. 문제 2에서 재사용합니다.